# Seattle Crime — Proximity Analysis

Explore SPD crime incidents **near any Seattle address**: enter an address, the
notebook geocodes it, filters every reported crime within a chosen radius, and
renders a dashboard — a full-window **crimes-per-year** trend, then **top offenses**
and **time of day** (both as a **% of the period**), a **proportional-symbol location
map** (dot size = incidents/year), and a **per-year density heatmap** — each compared
side by side for the **past 5 years vs the past year**, so you can see how the local
pattern is shifting without longer windows simply looking busier.

See [`README.md`](README.md) for setup (`uv sync`) and project details.

**Run the cells top to bottom:**

1. **Load** — auto-downloads the SPD CSV into `data/` on first run, then
   loads it (skipped if `df` is already in the kernel). As of June 2026 this is more than 337 MB and it will keep growing.
2. **Filter** — keep only the last `YEARS_BACK` years (default 10; this also drops
   the `1900-01-01` sentinels), then clip coordinates to a Seattle bbox.
3. **Functions** — define `geocode`, distance, and `analyze` helpers.
4. **Run** — running the last cell prompts you for an address (type or paste;
   blank = Space Needle). The dashboard renders inline and is saved to
   `charts/<address>.png`.

To analyze another address, just re-run that last cell and enter a new one.


In [ ]:
import pandas as pd
import urllib.request
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
# Socrata bulk CSV export for SPD Crime Data, 2008-Present (dataset tazs-3rd5)
DATASET_URL = "https://data.seattle.gov/api/views/tazs-3rd5/rows.csv?accessType=DOWNLOAD"

# Use any existing SPD CSV in data/; otherwise download it once (~400MB).
existing = sorted(DATA_DIR.glob("SPD_Crime_Data*.csv"))
if existing:
    csv_path = existing[-1]
else:
    csv_path = DATA_DIR / "SPD_Crime_Data__2008-Present.csv"
    print(f"No CSV found in {DATA_DIR}/ - downloading ~400MB (one time)...")

    # The server omits Content-Length (chunked/gzip), so report MB downloaded,
    # printing only every ~25 MB to avoid flooding the notebook output.
    _last = [0]

    def _progress(block, block_size, total):
        mb = block * block_size / 1e6
        if mb - _last[0] >= 25:
            _last[0] = mb
            print(f"  downloaded {mb:,.0f} MB...")

    urllib.request.urlretrieve(DATASET_URL, csv_path, reporthook=_progress)
    print(f"saved -> {csv_path}")


def parse_datetimes(s: pd.Series) -> pd.Series:
    """Parse SPD datetime strings. The Socrata API export uses MM/DD/YYYY; older
    manual exports use 'YYYY Mon DD'. Try the API format, fall back if it misses."""
    parsed = pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    if parsed.isna().mean() > 0.5:
        parsed = pd.to_datetime(s, format="%Y %b %d %I:%M:%S %p", errors="coerce")
    return parsed


# Skip the ~400MB read if df is already loaded in this kernel.
if "df" in globals():
    print(f"df already loaded ({len(df):,} rows) - skipping CSV read")
else:
    df = pd.read_csv(csv_path, low_memory=False)
    for _col in ["Report DateTime", "Offense Date"]:
        df[_col] = parse_datetimes(df[_col])
    print(f"loaded {len(df):,} rows x {df.shape[1]} columns from {csv_path.name}")

df.head()

In [ ]:
# Keep only recent offenses. This also drops the sentinel 1900-01-01 dates
# (and any other pre-cutoff junk) since they fall outside the window.
YEARS_BACK = 10   # <-- easy to change: how many calendar years of history to include

# Snap to Jan 1 of (this year - YEARS_BACK) so the first year is complete
# (the current year is still partial since it isn't over yet).
cutoff = pd.Timestamp(year=pd.Timestamp.now().year - YEARS_BACK, month=1, day=1)
before = len(df)
df = df[df["Offense Date"] >= cutoff]
print(f"Keeping offenses since {cutoff:%Y-%m-%d}: "
      f"removed {before - len(df):,} rows; {len(df):,} remain")

In [ ]:
# --- Clean coordinates -------------------------------------------------
# Lat/Long are REDACTED for ~14.5% of rows and contain some junk geocodes
# (lat -1/90, lon +/-175). Coerce to numeric and clip to a Seattle bbox.
df["lat"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["lon"] = pd.to_numeric(df["Longitude"], errors="coerce")

SEATTLE_BBOX = dict(lat_min=47.48, lat_max=47.74, lon_min=-122.46, lon_max=-122.22)
in_seattle = (
    df["lat"].between(SEATTLE_BBOX["lat_min"], SEATTLE_BBOX["lat_max"])
    & df["lon"].between(SEATTLE_BBOX["lon_min"], SEATTLE_BBOX["lon_max"])
)
geo = df[in_seattle].copy()
print(f"{len(geo):,} rows have usable Seattle coordinates "
      f"({len(geo)/len(df)*100:.1f}% of {len(df):,})")

In [ ]:
# --- Analysis functions ------------------------------------------------
# geocode an address -> filter incidents within a radius -> draw + save charts.
# Called from the last cell via analyze(ADDRESS, RADIUS_MILES).
import re
import urllib.parse
import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt


def geocode(address: str) -> tuple[float, float]:
    """Geocode a free-form address via OpenStreetMap Nominatim (nudged to Seattle)."""
    query = address if "seattle" in address.lower() else f"{address}, Seattle, WA"
    url = "https://nominatim.openstreetmap.org/search?" + urllib.parse.urlencode(
        {"q": query, "format": "json", "limit": 1}
    )
    req = urllib.request.Request(url, headers={"User-Agent": "seattle-crime-notebook"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        results = json.load(resp)
    if not results:
        raise ValueError(f"No geocoding result for {address!r}")
    return float(results[0]["lat"]), float(results[0]["lon"])


def haversine_miles(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles (vectorized)."""
    R = 3958.8  # Earth radius, miles
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def _short(label: str, n: int = 24) -> str:
    return label if len(label) <= n else label[: n - 1] + "…"


def _slug(text: str) -> str:
    """Filesystem-safe filename from a free-form address (handles '/', etc.)."""
    return re.sub(r"[^A-Za-z0-9,.-]+", "_", text).strip("_") or "address"


def analyze(address: str, radius_miles: float = 0.5):
    """Geocode `address`, keep crimes within `radius_miles`, draw + save the dashboard.

    Rows: crimes-per-year (full width), then - compared across the PAST 5 YEARS and
    PAST YEAR as columns - top offenses (% of the period), by hour of day (% of the
    period), a proportional-symbol location map (dot size = incidents/year), and a
    per-year density heatmap on a shared color scale. Percent-normalizing makes the
    shorter window directly comparable to the longer one.
    Exposes `nearby`, `home_lat`, `home_lon` as globals for further exploration.
    """
    global nearby, home_lat, home_lon
    home_lat, home_lon = geocode(address)
    print(f"{address} -> lat {home_lat:.5f}, lon {home_lon:.5f}")

    g = geo.copy()
    g["dist_mi"] = haversine_miles(g["lat"], g["lon"], home_lat, home_lon)
    nearby = g[g["dist_mi"] <= radius_miles].copy()
    print(f"{len(nearby):,} crimes within {radius_miles} mi of {address}")
    if nearby.empty:
        print("No incidents found - try a Seattle address or a larger radius.")
        return nearby

    now = pd.Timestamp.now()
    periods = [("Past 5 years", 5), ("Past year", 1)]
    subs = [(label, yrs, nearby[nearby["Offense Date"] >= now - pd.DateOffset(years=yrs)])
            for label, yrs in periods]
    ncols = len(periods)

    # Offenses: compare the SAME categories (overall top 10) as % of each period,
    # in the same order, so the bars line up column to column.
    top_cats = nearby["Offense Sub Category"].value_counts().head(10).index.tolist()
    offense_pct = [sub["Offense Sub Category"].value_counts(normalize=True)
                      .mul(100).reindex(top_cats).fillna(0) for _, _, sub in subs]
    off_xmax = max((p.max() for p in offense_pct), default=1.0)

    # Hour of day as % of each period (shared y so shapes/levels compare directly).
    hour_pct = []
    for _, _, sub in subs:
        hp = sub.groupby(sub["Offense Date"].dt.hour).size().reindex(range(24)).fillna(0)
        hour_pct.append(hp / hp.sum() * 100 if hp.sum() else hp)
    hour_ymax = max((p.max() for p in hour_pct), default=1.0)

    # Locations: collapse to unique block points, size by incidents/year (shared
    # scale) - avoids the blotchy overplotting of one dot per incident.
    loc_aggs, rate_max = [], 0.0
    for _, yrs, sub in subs:
        agg = sub.groupby(["lon", "lat"]).size().reset_index(name="cnt")
        agg["rate"] = agg["cnt"] / yrs
        loc_aggs.append(agg)
        rate_max = max(rate_max, float(agg["rate"].max()) if len(agg) else 0.0)
    rate_max = rate_max or 1.0

    pad = 0.1 * ((nearby["lon"].max() - nearby["lon"].min()) or 0.01)
    extent = (nearby["lon"].min() - pad, nearby["lon"].max() + pad,
              nearby["lat"].min() - pad, nearby["lat"].max() + pad)

    fig = plt.figure(figsize=(13, 22), constrained_layout=True)
    fig.suptitle(f"Crime near {address}  (within {radius_miles} mi)",
                 fontsize=16, weight="bold")
    gs = fig.add_gridspec(6, ncols, height_ratios=[0.8, 0.12, 1.0, 0.9, 1.15, 1.15])

    # Row 0 (full width): crimes per year, full window.
    ax = fig.add_subplot(gs[0, :])
    by_year = nearby.groupby(nearby["Offense Date"].dt.year).size()
    ax.bar(by_year.index, by_year.values, color="steelblue")
    ax.set_title("Crimes per year (full window; current year is partial)")
    ax.set_xlabel("year"); ax.set_ylabel("count")

    # Row 1: bold column headers naming each time window.
    for col, (label, yrs, sub) in enumerate(subs):
        hax = fig.add_subplot(gs[1, col]); hax.axis("off")
        hax.text(0.5, 0.4, f"{label.upper()}\n(n={len(sub):,})", ha="center",
                 va="center", fontsize=15, fontweight="bold")

    density_axes, hexbins = [], []
    for col, (label, yrs, sub) in enumerate(subs):
        # Row 2: top offenses as % of the period (aligned categories)
        ax = fig.add_subplot(gs[2, col])
        ax.set_title("Top offenses (% of period)", fontsize=11)
        cats_rev = top_cats[::-1]
        ax.barh([_short(c) for c in cats_rev],
                [offense_pct[col][c] for c in cats_rev], color="indianred")
        ax.set_xlim(0, off_xmax * 1.1); ax.set_xlabel("% of incidents")
        ax.tick_params(axis="y", labelsize=7)

        # Row 3: by hour of day as % of the period (shared y)
        ax = fig.add_subplot(gs[3, col])
        ax.set_title("By hour of day (% of period)", fontsize=11)
        hp = hour_pct[col]
        ax.plot(hp.index, hp.values, marker="o", color="seagreen")
        ax.set_ylim(0, hour_ymax * 1.1)
        ax.set_xlabel("hour"); ax.set_ylabel("% of incidents"); ax.set_xticks(range(0, 24, 6))

        # Row 4: proportional-symbol location map (dot area = incidents/year)
        ax = fig.add_subplot(gs[4, col])
        ax.set_title("Locations (dot size = incidents/yr)", fontsize=11)
        agg = loc_aggs[col]
        if len(agg):
            sizes = np.clip(agg["rate"] / rate_max * 170, 4, None)
            ax.scatter(agg["lon"], agg["lat"], s=sizes, alpha=0.45,
                       color="steelblue", edgecolor="none")
        ax.scatter([home_lon], [home_lat], s=140, color="red", marker="*",
                   edgecolor="white", linewidth=0.6)
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.tick_params(labelsize=6)

        # Row 5: per-year density heatmap (light cmap; shared, percentile-clipped scale)
        ax = fig.add_subplot(gs[5, col])
        ax.set_title("Density / year", fontsize=11)
        if not sub.empty:
            hb = ax.hexbin(sub["lon"], sub["lat"], C=np.full(len(sub), 1.0 / yrs),
                           reduce_C_function=np.sum, gridsize=40, extent=extent,
                           cmap="YlOrRd", mincnt=1)
            hexbins.append(hb)
        ax.scatter([home_lon], [home_lat], s=140, color="red", marker="*",
                   edgecolor="black", linewidth=0.6)
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.tick_params(labelsize=6)
        density_axes.append(ax)

    # Shared color scale clipped at the 97th percentile so one hotspot doesn't
    # darken everything; single horizontal colorbar under the row.
    if hexbins:
        allvals = np.concatenate([np.asarray(hb.get_array()) for hb in hexbins])
        vmax = float(np.percentile(allvals, 97)) if allvals.size else 1.0
        vmax = vmax or float(allvals.max() or 1.0)
        for hb in hexbins:
            hb.set_clim(0, vmax)
        fig.colorbar(hexbins[-1], ax=density_axes, location="bottom", shrink=0.6,
                     aspect=40, pad=0.02, extend="max", label="incidents per year")

    charts_dir = Path("charts")
    charts_dir.mkdir(exist_ok=True)
    outfile = charts_dir / (_slug(address) + ".png")
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    print(f"saved chart -> {outfile}")
    plt.show()
    return nearby

In [ ]:
# --- Run the analysis --------------------------------------------------
# Run this cell, then type or paste a Seattle address at the prompt
# (blank = Space Needle). input() blocks, so "Run All" waits here for you.
# The dashboard renders below and is saved to charts/<address>.png.
ADDRESS = input("Seattle address [Space Needle]: ").strip() or "Space Needle"
RADIUS_MILES = 0.5   # 0.25 = ~4 blocks, 0.5 = ~10 min walk, 1.0 = neighborhood

analyze(ADDRESS, RADIUS_MILES)